# Extract high-confidence demo patches

Uses TIAToolbox ResNet-18 (the same model used in production) to scan the PCam
test set and select patches where the model is >90% confident and correct.

No model upload needed — downloads directly from HuggingFace Hub.

**Dataset required:** andrewmvd/metastatic-tissue-classification-patchcamelyon

In [ ]:
!pip install -q timm

In [ ]:
import h5py
import numpy as np
from PIL import Image
import zipfile, os, torch
import timm

BASE = '/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon'

# TIAToolbox ResNet-18 — same weights used in production serving
# class 0 = tumour, class 1 = normal
model = timm.create_model("hf-hub:1aurent/resnet18.tiatoolbox-pcam", pretrained=True).eval()
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

def predict(patch_array):
    img = Image.fromarray(patch_array).convert("RGB")
    tensor = transform(img).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).squeeze()
    return float(probs[0])  # P(tumour) — class 0 is tumour in TIAToolbox

print("Model loaded")

In [ ]:
x_file = f'{BASE}/pcam/test_split.h5'
y_file = f'{BASE}/Labels/Labels/camelyonpatch_level_2_split_test_y.h5'

with h5py.File(y_file, 'r') as f:
    labels = f['y'][:].flatten()

print(f'Test set: {len(labels)} patches — {(labels==0).sum()} normal, {(labels==1).sum()} tumour')

SCAN     = 3000
CONF_MIN = 0.90   # only keep patches where model is >90% confident

normal_picks = []   # (idx, prob_tumour, confidence)
tumour_picks = []

with h5py.File(x_file, 'r') as f:
    imgs = f['x']
    for i in range(SCAN):
        label = labels[i]
        prob  = predict(imgs[i])
        pred  = int(prob >= 0.5)  # 0.5 threshold
        conf  = prob if pred == 1 else 1.0 - prob

        if pred != label or conf < CONF_MIN:
            continue

        if label == 0:
            normal_picks.append((i, prob, conf))
        else:
            tumour_picks.append((i, prob, conf))

        if len(normal_picks) >= 20 and len(tumour_picks) >= 20:
            break

        if i % 500 == 0:
            print(f'Scanned {i}/{SCAN} — normal: {len(normal_picks)}, tumour: {len(tumour_picks)}')

print(f'Found {len(normal_picks)} normal, {len(tumour_picks)} tumour high-confidence patches')

In [ ]:
os.makedirs('demo_patches', exist_ok=True)

normal_picks.sort(key=lambda x: x[2], reverse=True)
tumour_picks.sort(key=lambda x: x[2], reverse=True)

with h5py.File(x_file, 'r') as f:
    imgs = f['x']

    for i, (idx, prob, conf) in enumerate(normal_picks[:4], 1):
        patch = imgs[idx]
        Image.fromarray(patch).save(f'demo_patches/normal_{i}.png')
        print(f'normal_{i}.png  prob_tumour={prob:.3f}  confidence={conf:.3f}')
        display(Image.fromarray(patch))

    for i, (idx, prob, conf) in enumerate(tumour_picks[:4], 1):
        patch = imgs[idx]
        Image.fromarray(patch).save(f'demo_patches/tumour_{i}.png')
        print(f'tumour_{i}.png  prob_tumour={prob:.3f}  confidence={conf:.3f}')
        display(Image.fromarray(patch))

In [ ]:
with zipfile.ZipFile('demo_patches.zip', 'w') as zf:
    for f in os.listdir('demo_patches'):
        zf.write(f'demo_patches/{f}', f)

print('Done — download demo_patches.zip from the output panel')